# Foundry Weather Agent

This notebook creates a Microsoft Foundry weather agent that uses Open-Meteo for live location weather and provides clothing, activity, and safety recommendations. Authentication uses `DefaultAzureCredential` and the existing `az login` session.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition

# Find the .env file in this folder or one of its parents.
def find_env_file() -> Path:
    for directory in [Path.cwd(), *Path.cwd().parents]:
        candidate = directory / ".env"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Could not find .env in the current directory or a parent directory.")

env_file = find_env_file()
load_dotenv(env_file)
foundry_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"]
model_name = os.environ["MODEL_DEPLOYMENT_NAME"]

credential = DefaultAzureCredential()
foundry_client = AIProjectClient(endpoint=foundry_endpoint, credential=credential)
chat_client = foundry_client.get_openai_client()
print(f"Loaded {env_file}")
print(f"Using model deployment: {model_name}")

Loaded c:\Azure AI\AI_For_The_Azure_Practice_Entry_Level\Code\WeatherAgent\.env
Using model deployment: gpt-5-mini


In [ ]:
geocoding_openapi_spec = {
    "openapi": "3.1.0",
    "info": {
        "title": "Open-Meteo Geocoding API",
        "version": "1.0.0",
        "description": "Find coordinates for a city or place.",
    },
    "servers": [{"url": "https://geocoding-api.open-meteo.com"}],
    "paths": {
        "/v1/search": {
            "get": {
                "operationId": "geocode_location",
                "summary": "Find coordinates for a city or place",
                "parameters": [
                    {"name": "name", "in": "query", "required": True, "schema": {"type": "string"}},
                    {"name": "count", "in": "query", "schema": {"type": "integer", "default": 1, "maximum": 5}},
                    {"name": "language", "in": "query", "schema": {"type": "string", "default": "en"}},
                    {"name": "format", "in": "query", "schema": {"type": "string", "default": "json"}},
                ],
                "responses": {
                    "200": {
                        "description": "Matching locations",
                        "content": {"application/json": {"schema": {"type": "object"}}},
                    }
                },
            }
        }
    },
}

forecast_openapi_spec = {
    "openapi": "3.1.0",
    "info": {
        "title": "Open-Meteo Forecast API",
        "version": "1.0.0",
        "description": "Retrieve current and forecast weather.",
    },
    "servers": [{"url": "https://api.open-meteo.com"}],
    "paths": {
        "/v1/forecast": {
            "get": {
                "operationId": "get_weather_forecast",
                "summary": "Get current and forecast weather",
                "parameters": [
                    {"name": "latitude", "in": "query", "required": True, "schema": {"type": "number"}},
                    {"name": "longitude", "in": "query", "required": True, "schema": {"type": "number"}},
                    {"name": "current", "in": "query", "schema": {"type": "string", "default": "temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,showers,snowfall,weather_code,cloud_cover,wind_speed_10m,wind_gusts_10m"}},
                    {"name": "hourly", "in": "query", "schema": {"type": "string", "default": "temperature_2m,apparent_temperature,precipitation_probability,weather_code,wind_speed_10m"}},
                    {"name": "forecast_days", "in": "query", "schema": {"type": "integer", "default": 3, "minimum": 1, "maximum": 7}},
                    {"name": "timezone", "in": "query", "schema": {"type": "string", "default": "auto"}},
                    {"name": "temperature_unit", "in": "query", "schema": {"type": "string", "enum": ["celsius", "fahrenheit"], "default": "celsius"}},
                ],
                "responses": {
                    "200": {
                        "description": "Weather observations and forecast",
                        "content": {"application/json": {"schema": {"type": "object"}}},
                    }
                },
            }
        }
    },
}

def openapi_tool(name: str, spec: dict) -> dict:
    return {
        "type": "openapi",
        "openapi": {
            "name": name,
            "spec": spec,
            "auth": {"type": "anonymous"},
        },
    }

geocoding_tool = openapi_tool("open_meteo_geocoding", geocoding_openapi_spec)
forecast_tool = openapi_tool("open_meteo_forecast", forecast_openapi_spec)
print("Weather API tools configured.")

Weather API tool configured.


In [ ]:
weather_agent_name = "weather-activity-advisor"
weather_instructions = """
You are a practical weather and activity advisor.

For every location-specific request, use the Open-Meteo tools to geocode the location and retrieve current or forecast weather before answering. Never invent weather values. State the location, local time or forecast period, and the relevant retrieved conditions.

Give concise, practical recommendations for clothing, outdoor or indoor activities, and safety. Consider temperature, feels-like temperature, rain probability, precipitation, snow, wind, and daylight. Recommend layers, waterproof clothing, sun protection, hydration, or indoor alternatives when appropriate. For dangerous or severe conditions, advise checking official local alerts. Ask a clarifying question if the location or time period is missing or ambiguous.
"""

weather_agent = foundry_client.agents.create_version(
    agent_name=weather_agent_name,
    definition=PromptAgentDefinition(
        model=model_name,
        instructions=weather_instructions,
        tools=[geocoding_tool, forecast_tool],
    ),
)
print(f"Weather agent ready: {weather_agent.name} (version {weather_agent.version})")

Weather agent ready: weather-activity-advisor (version 3)


In [14]:
chat_session = chat_client.conversations.create()


def ask_weather_agent(question: str) -> str:
    result = chat_client.responses.create(
        conversation=chat_session.id,
        extra_body={
            "agent": {
                "name": weather_agent_name,
                "type": "agent_reference",
            }
        },
        input=question,
    )
    return result.output_text

print(ask_weather_agent("What should I wear and do outside in Stockholm today evening?"))

BadRequestError: Error code: 400 - {'error': {'message': "Invalid OpenAPI specification: ('Invalid OpenAPI specification format. See https://swagger.io/specification/ for details.', {'openapi': '3.1.0', 'info': {'title': 'Open-Meteo Weather API', 'version': '1.0.0', 'description': 'Resolve locations and retrieve current or forecast weather.'}, 'paths': {'/v1/search': {'servers': [{'url': 'https://geocoding-api.open-meteo.com'}], 'get': {'operationId': 'geocode_location', 'summary': 'Find coordinates for a city or place', 'parameters': [{'name': 'name', 'in': 'query', 'required': True, 'schema': {'type': 'string'}}, {'name': 'count', 'in': 'query', 'schema': {'type': 'integer', 'default': 1, 'maximum': 5}}, {'name': 'language', 'in': 'query', 'schema': {'type': 'string', 'default': 'en'}}, {'name': 'format', 'in': 'query', 'schema': {'type': 'string', 'default': 'json'}}], 'responses': {'200': {'description': 'Matching locations', 'content': {'application/json': {'schema': {'type': 'obj

## Test prompts

Run additional questions with `ask_weather_agent(...)`:

- `What should I wear in London tomorrow morning?`
- `Is this afternoon suitable for cycling in Amsterdam?`
- `Suggest indoor activities in New York if it rains tomorrow.`
- `What is the weather in Springfield?` to test ambiguous locations.

Weather information is advisory and should not replace official emergency or severe-weather alerts.